# Feature Selection

## Quick Overview
**Feature Selection** is the process of selecting a subset of relevant features (variables/columns) from your dataset to improve model performance, reduce training time, and avoid overfitting.

**Key Difference from Dimensionality Reduction:**
- **Feature Selection**: Keeps original features (subset of original)
- **Dimensionality Reduction** (PCA, t-SNE, UMAP): Creates new transformed features

## Why Feature Selection Matters

### Benefits
✅ **Improved Model Performance** - Less noise, focus on signal  
✅ **Faster Training** - Fewer features = less computation  
✅ **Reduced Overfitting** - Simpler models generalize better  
✅ **Lower Cost** - Fewer features to collect/store  
✅ **Better Interpretability** - Fewer features to explain  
✅ **Avoid Curse of Dimensionality** - Too many features, too few samples  

### Curse of Dimensionality
With more features than samples:
- Models become hard to train
- Performance gets worse
- Overfitting increases dramatically
- Example: 1000 features × 100 samples → 10 features × 100 samples is better

## Three Main Categories of Feature Selection

### 1. Filter Methods ⏭️
- Evaluate features **independently** of the model
- Fast and simple
- Good for initial screening
- **Methods**: Correlation, Variance, Chi-square, Mutual Information, ANOVA

| Pros | Cons |
|------|------|
| ⚡ Very fast | ❌ Ignores feature interactions |
| 💡 Model-agnostic | ❌ May miss useful combinations |
| 🎯 Easy to interpret | ❌ Less accurate than other methods |

### 2. Wrapper Methods 🔄
- Evaluate features using **actual model performance**
- Treats feature selection as search problem
- Slower but more accurate
- **Methods**: Forward Selection, Backward Elimination, RFE

| Pros | Cons |
|------|------|
| ✅ Uses actual model performance | 🐢 Computationally expensive |
| ✅ Captures feature interactions | 💔 Risk of overfitting |
| ✅ Best for small feature sets | ❌ Slow for large datasets |

### 3. Embedded Methods 🔗
- Feature selection **during model training**
- Fast and model-specific
- Balanced approach
- **Methods**: L1 Regularization (Lasso), Tree Feature Importance, Coefficients

| Pros | Cons |
|------|------|
| ⚡ Fast | ❌ Model-dependent |
| ✅ Captures feature interactions | ❌ Only works with some models |
| 🎯 Good balance | ❌ Less transparent |

## Comparison Table

| Method | Speed | Accuracy | Interactions | Good For |
|--------|-------|----------|--------------|----------|
| **Filter** (Correlation) | ⚡⚡⚡ | ⭐⭐ | ❌ | Quick screening |
| **Filter** (Mutual Info) | ⚡⚡ | ⭐⭐⭐ | ✅ | Non-linear patterns |
| **Wrapper** (RFE) | 🐢 | ⭐⭐⭐⭐ | ✅ | Small feature sets |
| **Embedded** (Lasso) | ⚡⚡ | ⭐⭐⭐⭐ | ✅ | Linear models |
| **Embedded** (Tree) | ⚡ | ⭐⭐⭐⭐ | ✅ | Tree-based models |

## Specific Techniques

### Filter Methods

#### 1. **Variance Threshold**
- Remove features with low variance (barely change)
- Good for quick cleanup
- Use when: Many features have constant values

#### 2. **Correlation with Target**
- Keep features most correlated with target variable
- Simple and interpretable
- Use when: Linear relationships expected

#### 3. **Chi-Square Test** (Classification, Categorical)
- Tests independence between feature and target
- P-value indicates statistical significance
- Use when: Working with categorical features

#### 4. **ANOVA F-Test** (Classification, Numeric)
- Tests if feature means differ across classes
- Higher F-score = more discriminative
- Use when: Multi-class classification

#### 5. **Mutual Information** (Classification)
- Measures shared information between feature and target
- Captures non-linear relationships
- Use when: Non-linear patterns expected

### Wrapper Methods

#### 1. **Forward Selection**
- Start with 0 features
- Iteratively add feature that improves model most
- Faster but greedy

#### 2. **Backward Elimination**
- Start with all features
- Iteratively remove feature that hurts least
- Computationally expensive

#### 3. **Recursive Feature Elimination (RFE)**
- Start with all features
- Train model, remove least important feature
- Repeat until desired number remains
- Works with any model

### Embedded Methods

#### 1. **L1 Regularization (Lasso)**
- Penalizes model coefficients
- Forces some coefficients to exactly 0
- Features with zero coefficient → remove

#### 2. **Tree Feature Importance**
- Built-in to tree models (Random Forest, XGBoost, etc.)
- Based on how much each feature decreases impurity
- Simple and fast

#### 3. **Permutation Importance**
- Shuffle each feature and check performance drop
- More reliable than tree importance
- Works with any model

## Practical Example 1: Filter Methods - Variance Threshold

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, f_classif, 
    RFE, SelectFromModel, chi2, mutual_info_classif
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Load dataset
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

print("Original dataset shape:", X.shape)
print("\nFeature names:")
print(X.columns.tolist())
print("\nFirst few rows:")
print(X.head())

# Calculate variance for each feature
variances = X.var()
print("\n" + "="*50)
print("Feature Variances:")
print("="*50)
for feature, var in variances.items():
    print(f"{feature:30s}: {var:.4f}")

# Apply Variance Threshold
print("\n" + "="*50)
print("Variance Threshold (threshold=0.5):")
print("="*50)
selector = VarianceThreshold(threshold=0.5)
X_filtered = selector.fit_transform(X)
selected_features = X.columns[selector.get_support()].tolist()

print(f"\nOriginal features: {X.shape[1]}")
print(f"Selected features: {X_filtered.shape[1]}")
print(f"Selected feature names: {selected_features}")
print(f"Removed features: {X.columns[~selector.get_support()].tolist()}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Variance plot
axes[0].barh(variances.index, variances.values, color=['green' if v > 0.5 else 'red' for v in variances.values])
axes[0].axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold')
axes[0].set_xlabel('Variance')
axes[0].set_title('Feature Variance (Threshold=0.5)', fontweight='bold')
axes[0].legend()

# Before and after
categories = ['Before', 'After']
counts = [X.shape[1], X_filtered.shape[1]]
colors = ['#FF6B6B', '#51CF66']
axes[1].bar(categories, counts, color=colors, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Number of Features')
axes[1].set_title('Feature Selection Results', fontweight='bold')
axes[1].set_ylim(0, 5)
for i, v in enumerate(counts):
    axes[1].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Insight: Variance threshold is fast but simple - removes constant/near-constant features only")

## Practical Example 2: Filter Methods - Correlation & SelectKBest

In [ ]:
# Load breast cancer dataset (more features for better demo)
bc_data = load_breast_cancer()
X_bc = pd.DataFrame(bc_data.data, columns=bc_data.feature_names)
y_bc = bc_data.target

print("Breast Cancer Dataset shape:", X_bc.shape)
print("Number of features:", X_bc.shape[1])

# Method 1: Correlation with Target
print("\n" + "="*60)
print("METHOD 1: CORRELATION WITH TARGET")
print("="*60)

# Convert target to numeric for correlation
correlations = []
for col in X_bc.columns:
    corr = X_bc[col].corr(pd.Series(y_bc))
    correlations.append(abs(corr))

correlations_df = pd.DataFrame({
    'Feature': X_bc.columns,
    'Correlation': correlations
}).sort_values('Correlation', ascending=False)

print("\nTop 5 Most Correlated Features:")
print(correlations_df.head(10))

# Keep top 5 features by correlation
top_k = 5
top_features = correlations_df.head(top_k)['Feature'].tolist()
print(f"\nTop {top_k} features by correlation: {top_features}")

# Method 2: SelectKBest with f_classif (ANOVA F-test)
print("\n" + "="*60)
print("METHOD 2: SELECTKBEST WITH F-CLASSIF (ANOVA)")
print("="*60)

selector_kbest = SelectKBest(score_func=f_classif, k=5)
X_kbest = selector_kbest.fit_transform(X_bc, y_bc)
selected_indices = selector_kbest.get_support(indices=True)
selected_features_kbest = X_bc.columns[selected_indices].tolist()

print(f"\nSelected features: {selected_features_kbest}")

# Get scores
scores = selector_kbest.scores_
feature_scores = pd.DataFrame({
    'Feature': X_bc.columns,
    'Score': scores
}).sort_values('Score', ascending=False)

print("\nTop 10 Features by F-Score:")
print(feature_scores.head(10))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Correlation plot
top_10_corr = correlations_df.head(10)
axes[0].barh(range(len(top_10_corr)), top_10_corr['Correlation'].values, color='steelblue')
axes[0].set_yticks(range(len(top_10_corr)))
axes[0].set_yticklabels(top_10_corr['Feature'].values, fontsize=9)
axes[0].set_xlabel('Absolute Correlation with Target')
axes[0].set_title('Top 10 Features by Correlation', fontweight='bold')
axes[0].invert_yaxis()

# F-Score plot
top_10_scores = feature_scores.head(10)
axes[1].barh(range(len(top_10_scores)), top_10_scores['Score'].values, color='coral')
axes[1].set_yticks(range(len(top_10_scores)))
axes[1].set_yticklabels(top_10_scores['Feature'].values, fontsize=9)
axes[1].set_xlabel('F-Score (ANOVA)')
axes[1].set_title('Top 10 Features by F-Score', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\n💡 Insight: SelectKBest is more sophisticated than correlation")
print("   - Correlation only captures linear relationships")
print("   - F-Score (ANOVA) captures how well features discriminate between classes")

## Practical Example 3: Wrapper Methods - Recursive Feature Elimination (RFE)

In [ ]:
# RFE - Recursive Feature Elimination
print("="*60)
print("WRAPPER METHOD: RECURSIVE FEATURE ELIMINATION (RFE)")
print("="*60)

# Split data for evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# RFE with Logistic Regression
print("\nApplying RFE with LogisticRegression...")
estimator = LogisticRegression(random_state=42, max_iter=1000)
rfe = RFE(estimator=estimator, n_features_to_select=5, step=1)
X_train_rfe = rfe.fit_transform(X_train_scaled, y_train)
X_test_rfe = rfe.transform(X_test_scaled)

selected_features_rfe = X_bc.columns[rfe.get_support()].tolist()
print(f"\nSelected {rfe.n_features_to_select} features via RFE:")
print(selected_features_rfe)

# Get feature rankings
rfe_ranking = pd.DataFrame({
    'Feature': X_bc.columns,
    'Ranking': rfe.ranking_
}).sort_values('Ranking')

print("\nFeature Rankings (1 = selected):")
print(rfe_ranking.head(10))

# Train models and compare performance
print("\n" + "="*60)
print("MODEL PERFORMANCE COMPARISON")
print("="*60)

# Model 1: All features
print("\n1. Logistic Regression with ALL features:")
model_all = LogisticRegression(random_state=42, max_iter=1000)
model_all.fit(X_train_scaled, y_train)
pred_all = model_all.predict(X_test_scaled)
acc_all = accuracy_score(y_test, pred_all)
print(f"   Accuracy: {acc_all:.4f}")
print(f"   Number of features: {X_train_scaled.shape[1]}")

# Model 2: RFE selected features
print("\n2. Logistic Regression with RFE-selected features:")
model_rfe = LogisticRegression(random_state=42, max_iter=1000)
model_rfe.fit(X_train_rfe, y_train)
pred_rfe = model_rfe.predict(X_test_rfe)
acc_rfe = accuracy_score(y_test, pred_rfe)
print(f"   Accuracy: {acc_rfe:.4f}")
print(f"   Number of features: {X_train_rfe.shape[1]}")
print(f"   Accuracy change: {acc_rfe - acc_all:+.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RFE Rankings
top_20 = rfe_ranking.head(20)
colors = ['green' if r == 1 else 'lightcoral' for r in top_20['Ranking'].values]
axes[0].barh(range(len(top_20)), top_20['Ranking'].values, color=colors)
axes[0].set_yticks(range(len(top_20)))
axes[0].set_yticklabels(top_20['Feature'].values, fontsize=9)
axes[0].set_xlabel('RFE Ranking (1=Selected)')
axes[0].set_title('RFE Feature Rankings', fontweight='bold')
axes[0].invert_yaxis()

# Accuracy comparison
methods = ['All Features\n({} feat)'.format(X_train_scaled.shape[1]), 
           'RFE Selected\n(5 feat)']
accuracies = [acc_all, acc_rfe]
colors = ['#3498db', '#2ecc71']
bars = axes[1].bar(methods, accuracies, color=colors, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Model Performance Comparison', fontweight='bold')
axes[1].set_ylim([0.8, 1.0])
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Insight: RFE uses model performance to guide feature selection")
print("   - More accurate than filter methods")
print("   - Slower but captures feature interactions")
print("   - Often achieves same or better performance with fewer features")

## Practical Example 4: Embedded Methods - Tree Feature Importance

In [ ]:
print("="*60)
print("EMBEDDED METHOD: TREE FEATURE IMPORTANCE")
print("="*60)

# Train Random Forest and get feature importance
print("\nTraining Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Get feature importances
importances = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': X_bc.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nTop 10 Features by Tree Importance:")
print(feature_importance_df.head(10))

# Select top 5 features
threshold_importance = sorted(importances, reverse=True)[4]  # 5th feature
important_features = X_bc.columns[importances >= threshold_importance].tolist()
print(f"\nTop 5 Important Features:")
print(important_features)

# Train models with different feature subsets
X_train_top5 = X_train[important_features]
X_test_top5 = X_test[important_features]

# Model with all features
print("\n" + "="*60)
print("MODEL PERFORMANCE WITH RANDOM FOREST")
print("="*60)

print("\n1. Random Forest with ALL features:")
rf_all = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_all.fit(X_train, y_train)
pred_all_rf = rf_all.predict(X_test)
acc_all_rf = accuracy_score(y_test, pred_all_rf)
print(f"   Accuracy: {acc_all_rf:.4f}")
print(f"   Number of features: {X_train.shape[1]}")

# Model with top 5 features
print("\n2. Random Forest with TOP 5 features:")
rf_top5 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_top5.fit(X_train_top5, y_train)
pred_top5_rf = rf_top5.predict(X_test_top5)
acc_top5_rf = accuracy_score(y_test, pred_top5_rf)
print(f"   Accuracy: {acc_top5_rf:.4f}")
print(f"   Number of features: {X_train_top5.shape[1]}")
print(f"   Accuracy change: {acc_top5_rf - acc_all_rf:+.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature Importance
top_15 = feature_importance_df.head(15)
colors = ['green' if imp >= threshold_importance else 'lightcoral' for imp in top_15['Importance'].values]
axes[0].barh(range(len(top_15)), top_15['Importance'].values, color=colors)
axes[0].set_yticks(range(len(top_15)))
axes[0].set_yticklabels(top_15['Feature'].values, fontsize=9)
axes[0].set_xlabel('Feature Importance')
axes[0].set_title('Random Forest Feature Importance', fontweight='bold')
axes[0].axvline(x=threshold_importance, color='black', linestyle='--', linewidth=2, label='Top 5 threshold')
axes[0].legend()
axes[0].invert_yaxis()

# Accuracy comparison
methods = ['All Features\n({} feat)'.format(X_train.shape[1]), 
           'Top 5 Important\n(5 feat)']
accuracies = [acc_all_rf, acc_top5_rf]
colors = ['#3498db', '#2ecc71']
bars = axes[1].bar(methods, accuracies, color=colors, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Random Forest Performance', fontweight='bold')
axes[1].set_ylim([0.8, 1.0])
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Insight: Tree Feature Importance is fast and interpretable")
print("   - Built-in to tree-based models")
print("   - Shows which features tree uses for splits")
print("   - Often matches RFE results with much faster computation")

## Practical Example 5: Method Comparison

In [ ]:
# Compare all methods on same dataset
print("="*60)
print("COMPREHENSIVE METHOD COMPARISON")
print("="*60)

import time

results = []

# 1. Variance Threshold
print("\n1. Variance Threshold...")
start = time.time()
vt = VarianceThreshold(threshold=0.1)
X_vt = vt.fit_transform(X_train)
time_vt = time.time() - start
model_vt = LogisticRegression(random_state=42, max_iter=1000)
model_vt.fit(X_vt, y_train)
acc_vt = model_vt.score(vt.transform(X_test), y_test)
results.append({
    'Method': 'Variance Threshold',
    'Features Selected': X_vt.shape[1],
    'Accuracy': acc_vt,
    'Time (s)': time_vt,
    'Type': 'Filter'
})

# 2. SelectKBest (F-classif)
print("2. SelectKBest (F-classif)...")
start = time.time()
skb = SelectKBest(f_classif, k=5)
X_skb = skb.fit_transform(X_train, y_train)
time_skb = time.time() - start
model_skb = LogisticRegression(random_state=42, max_iter=1000)
model_skb.fit(X_skb, y_train)
acc_skb = model_skb.score(skb.transform(X_test), y_test)
results.append({
    'Method': 'SelectKBest (F-classif)',
    'Features Selected': X_skb.shape[1],
    'Accuracy': acc_skb,
    'Time (s)': time_skb,
    'Type': 'Filter'
})

# 3. Mutual Information
print("3. SelectKBest (Mutual Info)...")
start = time.time()
smi = SelectKBest(mutual_info_classif, k=5)
X_smi = smi.fit_transform(X_train, y_train)
time_smi = time.time() - start
model_smi = LogisticRegression(random_state=42, max_iter=1000)
model_smi.fit(X_smi, y_train)
acc_smi = model_smi.score(smi.transform(X_test), y_test)
results.append({
    'Method': 'Mutual Information',
    'Features Selected': X_smi.shape[1],
    'Accuracy': acc_smi,
    'Time (s)': time_smi,
    'Type': 'Filter'
})

# 4. RFE
print("4. RFE...")
start = time.time()
rfe_comp = RFE(LogisticRegression(random_state=42, max_iter=1000), n_features_to_select=5)
X_rfe_comp = rfe_comp.fit_transform(X_train_scaled, y_train)
time_rfe = time.time() - start
model_rfe_comp = LogisticRegression(random_state=42, max_iter=1000)
model_rfe_comp.fit(X_rfe_comp, y_train)
acc_rfe_comp = model_rfe_comp.score(rfe_comp.transform(X_test_scaled), y_test)
results.append({
    'Method': 'RFE',
    'Features Selected': X_rfe_comp.shape[1],
    'Accuracy': acc_rfe_comp,
    'Time (s)': time_rfe,
    'Type': 'Wrapper'
})

# 5. Tree Feature Importance
print("5. Tree Importance...")
start = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
importances = rf.feature_importances_
top_5_indices = np.argsort(importances)[-5:]
X_tree = X_train.iloc[:, top_5_indices]
X_test_tree = X_test.iloc[:, top_5_indices]
time_tree = time.time() - start
model_tree = LogisticRegression(random_state=42, max_iter=1000)
model_tree.fit(X_tree, y_train)
acc_tree = model_tree.score(X_test_tree, y_test)
results.append({
    'Method': 'Tree Importance',
    'Features Selected': X_tree.shape[1],
    'Accuracy': acc_tree,
    'Time (s)': time_tree,
    'Type': 'Embedded'
})

# 6. All features (baseline)
print("6. Baseline (All features)...")
start = time.time()
model_baseline = LogisticRegression(random_state=42, max_iter=1000)
model_baseline.fit(X_train_scaled, y_train)
time_baseline = time.time() - start
acc_baseline = model_baseline.score(X_test_scaled, y_test)
results.append({
    'Method': 'All Features (Baseline)',
    'Features Selected': X_train_scaled.shape[1],
    'Accuracy': acc_baseline,
    'Time (s)': time_baseline,
    'Type': 'Baseline'
})

# Display results
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
print(results_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Accuracy comparison
colors = {'Filter': '#3498db', 'Wrapper': '#e74c3c', 'Embedded': '#2ecc71', 'Baseline': '#95a5a6'}
bar_colors = [colors[t] for t in results_df['Type']]
axes[0].barh(results_df['Method'], results_df['Accuracy'], color=bar_colors)
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Accuracy Comparison', fontweight='bold')
axes[0].set_xlim([0.8, 1.0])

# Features selected
axes[1].barh(results_df['Method'], results_df['Features Selected'], color=bar_colors)
axes[1].set_xlabel('Number of Features')
axes[1].set_title('Features Selected', fontweight='bold')

# Execution time
axes[2].barh(results_df['Method'], results_df['Time (s)'], color=bar_colors)
axes[2].set_xlabel('Time (seconds)')
axes[2].set_title('Execution Time', fontweight='bold')
axes[2].set_xscale('log')

plt.tight_layout()
plt.show()

print("\n💡 Insights:")
print("✓ Filter methods are fast but may miss interactions")
print("✓ Wrapper methods are slower but capture interactions")
print("✓ Embedded methods offer good balance of speed and accuracy")
print("✓ Often, fewer features = better generalization")

## Best Practices & Decision Guide

### When to Use Each Method

```
🎯 Decision Tree:

START: How many features do you have?

├─ 1,000+ features?
│  └─ Use Filter methods FIRST (variance, correlation)
│     Then apply Wrapper/Embedded on reduced set
│
├─ 100-1000 features?
│  ├─ Time critical? → Tree Feature Importance (fast)
│  └─ Accuracy critical? → RFE (slow but accurate)
│
├─ 10-100 features?
│  ├─ Linear model? → Lasso (L1 regularization)
│  ├─ Tree model? → Tree Feature Importance
│  └─ Any model? → RFE
│
└─ <10 features?
   └─ Maybe not needed! But if wanted: any method fine
```

### Step-by-Step Workflow

**Phase 1: Initial Screening (Fast)**
```python
# Remove low-variance features
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.01)
X_filtered = selector.fit_transform(X)
```

**Phase 2: Univariate Selection (Medium speed)**
```python
# Keep top-k most correlated with target
from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(f_classif, k=50)  # Reduce to 50 features
X_selected = selector.fit_transform(X, y)
```

**Phase 3: Fine Tuning (Based on time)**
```python
# Option A: Fast (Embedded)
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
rf = RandomForestClassifier(n_estimators=100, random_state=42)
selector = SelectFromModel(rf)
X_final = selector.fit_transform(X_selected, y)

# Option B: Accurate (Wrapper)
from sklearn.feature_selection import RFE
rfe = RFE(estimator, n_features_to_select=10)
X_final = rfe.fit_transform(X_selected, y)
```

### Common Mistakes to Avoid

❌ **Mistake 1: Selecting features on entire dataset**
```python
# WRONG - data leakage!
selector = SelectKBest(f_classif, k=10)
X_selected = selector.fit_transform(X, y)
X_train, X_test = train_test_split(X_selected, y)

# RIGHT - fit on train only
X_train, X_test, y_train, y_test = train_test_split(X, y)
selector = SelectKBest(f_classif, k=10)
selector.fit(X_train, y_train)
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)
```

❌ **Mistake 2: Not scaling features**
```python
# WRONG - features have different scales
selector = RFE(model, n_features_to_select=5)

# RIGHT - standardize first
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(X)
selector = RFE(model, n_features_to_select=5)
```

❌ **Mistake 3: Selecting too many or too few**
```python
# WRONG - removes all signal
k = 1  # Too few!

# WRONG - removes no noise
k = len(features)  # Too many!

# RIGHT - use rule of thumb or validation
k = max(5, len(features) // 10)  # At least 5, or 10% of features
```

❌ **Mistake 4: Over-relying on single method**
```python
# WRONG - only trusting one method
top_features = get_top_features_method_1()

# RIGHT - compare multiple methods
features_method1 = get_top_features_method_1()
features_method2 = get_top_features_method_2()
common_features = set(features_method1) & set(features_method2)
```

### Quick Reference: Method Selection

| Situation | Best Method | Why |
|-----------|------------|-----|
| **Need interpretability** | Tree Importance | Easy to explain |
| **Many features (1000+)** | Correlation + Filter | Fast initial screening |
| **Small feature set** | RFE | Most accurate |
| **Linear model** | Lasso (L1) | Built-in regularization |
| **Tree-based model** | Tree Importance | Native support |
| **Limited time** | Tree/Mutual Info | Fast computation |
| **Maximum accuracy** | RFE | Considers interactions |
| **Production pipeline** | Embedded (Lasso/Tree) | Consistent, reproducible |

## Key Takeaways 🎯

### What is Feature Selection?
Process of selecting most relevant subset of features to:
- Improve model performance
- Reduce training time
- Improve interpretability
- Avoid overfitting
- Handle curse of dimensionality

### Three Approaches
1. **Filter Methods** ⏭️ - Fast, simple, model-agnostic
2. **Wrapper Methods** 🔄 - Slow, accurate, captures interactions
3. **Embedded Methods** 🔗 - Balanced, model-dependent

### Key Algorithms

| Algorithm | Type | Speed | Use When |
|-----------|------|-------|----------|
| **Variance Threshold** | Filter | ⚡⚡⚡ | Quick cleanup |
| **Correlation** | Filter | ⚡⚡⚡ | Linear relationships |
| **ANOVA F-Test** | Filter | ⚡⚡ | Numeric classification |
| **Mutual Information** | Filter | ⚡ | Non-linear relationships |
| **RFE** | Wrapper | 🐢 | Small feature sets |
| **Lasso (L1)** | Embedded | ⚡⚡ | Linear models |
| **Tree Importance** | Embedded | ⚡⚡ | Tree-based models |

### Essential Rules
✅ Always split data before feature selection (no leakage)  
✅ Standardize features for most methods  
✅ Start with Filter methods for initial screening  
✅ Use domain knowledge + statistics  
✅ Compare multiple methods  
✅ Validate with cross-validation  
✅ Don't remove too many features (keep signal)  
✅ Monitor performance on test set  

### Common Results
- Usually 5-10% of original features work well
- Fewer features often = better generalization
- Often little accuracy loss with significant feature reduction
- Computational savings are substantial

### Mathematical Concepts
**Variance** = How much feature varies
$$\text{Var}(X) = E[(X - \mu)^2]$$

**Correlation** = Linear relationship strength
$$\text{Corr}(X, Y) = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y}$$

**F-Score** (ANOVA) = Between-class vs within-class variance
$$F = \frac{\text{Between-group variance}}{\text{Within-group variance}}$$

**Mutual Information** = Shared information between variables
$$I(X; Y) = \sum_x \sum_y p(x,y) \log \frac{p(x,y)}{p(x)p(y)}$$

## Implementation Patterns

### Pattern 1: Quick Feature Screening
```python
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

# Remove constant features
vt = VarianceThreshold()
X_var = vt.fit_transform(X)

# Keep top-k best
selector = SelectKBest(f_classif, k=min(20, X_var.shape[1]//5))
X_final = selector.fit_transform(X_var, y)
```

### Pattern 2: Production Pipeline
```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# Create pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', RFE(RandomForestClassifier(), n_features_to_select=10)),
    ('model', RandomForestClassifier())
])

# Fit and predict
pipe.fit(X_train, y_train)
predictions = pipe.predict(X_test)
```

### Pattern 3: Ensemble Feature Selection
```python
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier

# Multiple methods
methods = [
    ('f_classif', SelectKBest(f_classif, k=10)),
    ('mutual_info', SelectKBest(mutual_info_classif, k=10)),
    ('rfe', RFE(RandomForestClassifier(), n_features_to_select=10)),
]

# Get consensus features
all_selected = set()
for name, selector in methods:
    selector.fit(X, y)
    selected = set(X.columns[selector.get_support()])
    all_selected.update(selected)

# Keep only features selected by multiple methods
feature_votes = {}
for name, selector in methods:
    for feat in X.columns[selector.get_support()]:
        feature_votes[feat] = feature_votes.get(feat, 0) + 1

final_features = [f for f, votes in feature_votes.items() if votes >= 2]
```

## Further Resources

### Documentation
📖 [scikit-learn Feature Selection](https://scikit-learn.org/stable/modules/feature_selection.html)
📖 [Feature Selection Methods Overview](https://en.wikipedia.org/wiki/Feature_selection)

### Recommended Reading
📚 "An Introduction to Feature Selection" - Feature selection survey  
📚 "Feature Engineering for Machine Learning" - Zheng & Casari  
📚 "Machine Learning Interpretability" - Feature importance deep dive  

### Related Topics
- **Feature Engineering** - Creating better features
- **Dimensionality Reduction** - PCA, t-SNE, UMAP for visualization
- **Regularization** - L1/L2 penalties for implicit feature selection
- **Ensemble Methods** - Combine multiple feature selection approaches
- **Domain Expertise** - Often best source of feature ideas

### Quick Reference: All Methods

```python
# Import everything
from sklearn.feature_selection import (
    VarianceThreshold,         # Remove low variance
    SelectKBest,               # Select top-k
    f_classif, f_regression,   # F-statistics
    chi2,                      # Chi-square test
    mutual_info_classif,       # Mutual information
    RFE,                       # Recursive elimination
    SelectFromModel,           # Model-based selection
)

# Use with any model
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Create pipelines
from sklearn.pipeline import Pipeline, FeatureUnion
```

---

### 🎓 Next Steps
1. **Start with your data** - Load dataset and understand features
2. **Try Filter methods** - Quick baseline with variance/correlation
3. **Experiment with Wrapper** - RFE on reduced feature set
4. **Use Embedded methods** - Tree importance for speed
5. **Compare results** - Accuracy, feature count, training time
6. **Validate** - Cross-validation on multiple splits
7. **Deploy** - Save selected features for production
8. **Monitor** - Track feature performance over time